In [2]:
import os
from anthropic import Anthropic
import json
from pathlib import Path

### Milestone A

In [ ]:
def getJobData(keyword: str):
    keyword = keyword.lower() # normalize the keyword
    folder = Path("postings") # get filepath to the postings
    results = {}

    for path in folder.glob("*.txt"):
        results[path.name] = []
        text = path.read_text(encoding="utf-8").lower()
        i = 0
        end = len(text)
        # find all instances of the keyword
        while i != -1:
            pos = text.find(keyword, i, end)
            if pos != -1:
                if pos >= 60:
                    results[path.name].append(text[pos-60:pos+61])
                else:
                    results[path.name].append(text[0:pos+61])
                i = pos + len(keyword)
            else: i = -1
    return results

tools = [
    {
        "name": "get_jd_info",
        "description": "Search all of the user's saved job descriptions to get information about a particular skill or qualification needed for them. Ex: Needing to know PyTorch, Kubernetes, C++, etc. Using this tool will return a dictionary where the key is the filename and the value is a list of snippets containing the keyword from the JD. Note that 1 list corresponds to 1 job description, and a list can have 0, 1 or more snippets in it.  If a key has an empty list, this indicates that the keyword was not in the JD.",
        "input_schema": {
            "type": "object",
            "properties": {
                "keyword": {
                    "type": "string",
                    "description": "The skill/qualification you want to learn more about. Note, this tool only returns things based on keyword search, so you'll have to be specific about what you want. For instance, you cannot generalize and ask about AI-based skills and expect to get back an answer that talks about knowing LangGraph, how to build harnesses, etc. This is one case insensitive keyword/phrase per call. You'll get back matches or no matches with the keyword \"AI skills\". Example input for tool use: PyTorch. ",
                },
            },
            "required": ["keyword"],
        },
    }
]

client = Anthropic(
    api_key = os.environ.get("ANTHROPIC_API_KEY")
)

# 1st Response
messages = [{"role": "user", "content": "I have a bunch of job descriptions saved and I need to know which of them require/recommend that I know PyTorch."}]
result = client.messages.create(
    model = "claude-haiku-4-5-20251001",
    max_tokens = 1024,
    messages = messages,
    tools = tools,
    tool_choice = {"type": "auto", "disable_parallel_tool_use": True}
)

print("Response 1:", result)

tool_input = None
tool_block = None

while result.stop_reason == "tool_use":
    tool_block = None
    tool_input = None
    for block in result.content:
        if block.type == "tool_use":
            tool_block = block
            if tool_block.name == "get_jd_info":
                tool_input = block.input["keyword"]

    if tool_input == None:
        print("Claude requested an unknown tool: ", tool_block.name)
        print("Notifying Claude so it can re-request with the correct tool...")

        # add Claude's message to the list
        messages.append({"role": "assistant", "content": result.content})

        # add tool response to the list
        messages.append({"role":"user",
                            "content": [{
                                "type": "tool_result",
                                "tool_use_id": tool_block.id,
                                "content": "Unknown tool was called. Look at the available tools again carefully to make sure you're calling the correct tool!"
                            }]
                        })

    else:
        # call tool
        job_info = getJobData(tool_input)

        # need to convert tool result content to a string or list
        job_info = str(job_info)

        # add Claude's message to the list
        messages.append({"role": "assistant", "content": result.content})

        # add tool response to the list
        messages.append({"role":"user",
                            "content": [{
                                "type": "tool_result",
                                "tool_use_id": tool_block.id,
                                "content": job_info
                            }]
                        })

    result = client.messages.create(
        model = "claude-haiku-4-5-20251001",
        max_tokens = 1024,
        messages = messages,
        tools = tools,
        tool_choice = {"type": "auto", "disable_parallel_tool_use": True}
    )

final_ans = ""
for block in result.content:
    if block.type == "text":
        final_ans = block

print("Claude's final answer:", final_ans.text)


### Milestone B

In [26]:
def get_job_info(keyword: str):
    keyword = keyword.lower() # normalize the keyword
    folder = Path("postings") # get filepath to the postings
    results = {}

    for path in folder.glob("*.txt"):
        results[path.name] = []
        text = path.read_text(encoding="utf-8").lower()
        i = 0
        end = len(text)
        # find all instances of the keyword
        while i != -1:
            pos = text.find(keyword, i, end)
            if pos != -1:
                if pos >= 60:
                    results[path.name].append(text[pos-60:pos+61])
                else:
                    results[path.name].append(text[0:pos+61])
                i = pos + len(keyword)
            else: i = -1
    return results

def get_profile():
    # encoding = "utf-8" tells Python which rule to use when turning the file's bytes into characters. UTF-8 is the standard rule modern text uses.
    # It covers every character in every language, including special characters like ● and curly quotes.
    f = open("profile.md", encoding="utf-8")
    profile = f.read()
    f.close()
    return profile

def save_note(job_title: str, note_draft: str):
    print("\n")
    print(job_title)
    print(note_draft)
    return "Draft note printed successfully!"

def get_full_jd(job_title: str):
    path = "Postings/"+ job_title + ".txt"
    f = open(path, encoding="utf-8")
    jd = f.read()
    f.close()
    return jd

tools = [
    {
        "name": "get_job_info",
        "description": "Search all of the user's saved job descriptions to get information about a particular skill or qualification needed for them. Ex: Needing to know PyTorch, Kubernetes, C++, etc. Using this tool will return a dictionary where the key is the filename and the value is a list of snippets containing the keyword from the JD. Note that 1 list corresponds to 1 job description, and a list can have 0, 1 or more snippets in it.  If a key has an empty list, this indicates that the keyword was not in the JD.",
        "input_schema": {
            "type": "object",
            "properties": {
                "keyword": {
                    "type": "string",
                    "description": "The skill/qualification you want to learn more about. Note, this tool only returns things based on keyword search, so you'll have to be specific about what you want. For instance, you cannot generalize and ask about AI-based skills and expect to get back an answer that talks about knowing LangGraph, how to build harnesses, etc. This is one case insensitive keyword/phrase per call. You'll get back matches or no matches with the keyword \"AI skills\". Example input for tool use: PyTorch. ",
                },
            },
            "required": ["keyword"],
        },
    },

    {
        "name": "get_profile",
        "description": "Use this tool to view the user's profile: this includes her skill set, projects, timelines she's looking for, constraints/notes you should know, etc.",
        "input_schema": {
            "type": "object",
            "properties": {}
        },
    },

    {
        "name": "get_full_jd",
        "description": "Get the full job description of jobs you think the user is best qualified for.",
        "input_schema": {
            "type":"object",
            "properties":{
                "job_title": {
                    "type": "string",
                    "description": "Provide the title the EXACT SAME way it was listed in the get_job_info tool responses. Ex: DoorDash SWE"
                },
            },
            "required": ["job_title"]
        },
    },

    {
        "name": "save_note",
        "description": "Use this tool to return a first-draft message (brief cover note/outreach message) to the employer, grounded in information from the user's profile. This tool will print your first draft. You must title it based on the job posting the note is for.",
        "input_schema": {
            "type": "object",
            "properties":{
                "job_title":{
                    "type": "string",
                    "description": "the job posting that the draft-note is for. Ex: PIMCO Tech Analyst, SWE Draft Note."
                },
                "note_draft":{
                    "type": "string",
                    "description": "Add the actual note you've drafted here. Nothing more, nothing else"
                },
            },
            "required": ["job_title", "note_draft"]
        }
    }
]

client = Anthropic(
    api_key = os.environ.get("ANTHROPIC_API_KEY")
)

SYSTEM_PROMPT = "Your role is to help the user find jobs to apply to. Some of the tools provided give you access to (1) the user's profile (what they're looking for, experiences/projects, constraints, etc.) and (2) the job descriptions - these can only be viewed by keyword search to identify specific skills. To best do this job, first look at the user's profile and compare it with skills listed in the job descriptions. Once you get a feel for what the user is and isn't qualified for (based on projects, coursework, skills, etc), and identify the best suited jobs to apply for, you can request to see entire job descriptions to draft outreach note to the recruiter. All of the judgements, statements (including in the outreach message), and assumptions you make must be grounded in the user's profile and what you see in the job descriptions. Note that the output token limit for each turn is 1024 tokens, so keep that in mind as you're writing your final responses. You cannot pause to ask me questions because you're being run in a while stop_reason = tool_use loop."
messages = [{"role": "user", "content": "I have a bunch of job descriptions saved but need some help figuring out which ones I'm most qualified for. You have access to my profile via tool calls, so first go through it, find some skills that I have (Ex: Python, C++, MCP), and compare it with skills listed in the job descriptions. Based on the information you've gathered, tell me which jobs I'd be a good fit for and which I don't know enough for."}]
result = client.messages.create(
    model = "claude-haiku-4-5-20251001",
    max_tokens = 1024,
    system = SYSTEM_PROMPT,
    messages = messages,
    tools = tools,
    tool_choice = {"type": "auto", "disable_parallel_tool_use": True}
)

print("Response before entering the tol_use loop:", result)
i = 1

while result.stop_reason == "tool_use":

    # append Claude's request to the message list
    messages.append({"role": "assistant", "content": result.content})
    print(f"Iteration {i}, no. of input tokens: {result.usage.input_tokens}")

    tool_block = None
    jd_keyword = None
    job_title_save_note = None
    job_title_get_full_jd = None
    note_draft = None
    need_profile = False

    # Processing the tool_block and setting relevant parameters
    for block in result.content:
        if block.type == "tool_use":
            tool_block = block
            if tool_block.name == "get_job_info":
                jd_keyword = block.input["keyword"]

            elif tool_block.name == "save_note":
                job_title_save_note = block.input["job_title"]
                note_draft = block.input["note_draft"]

            elif tool_block.name == "get_profile":
                need_profile = True

            elif tool_block.name == "get_full_jd":
                job_title_get_full_jd = block.input["job_title"]

            else:
                print(f"Iteration {i}, Claude requested an unknown tool: ", tool_block.name)
                # Claude gets notified in the next if, elif, else block

    # get_job_data tool request
    if jd_keyword != None:
        print(f"Iteration {i}, calling get_job_info on keyword: {jd_keyword} \n")
        # call tool
        job_info = get_job_info(jd_keyword)

        # need to convert tool result content to a string or list
        job_info = str(job_info)

        # add tool response to the list
        messages.append({"role":"user",
                            "content": [{
                                "type": "tool_result",
                                "tool_use_id": tool_block.id,
                                "content": job_info
                            }]
                        })

    # save_note tool request
    elif job_title_save_note != None and note_draft != None:
        print(f"Iteration {i}, calling save_note \n")
        save_note_res = save_note(job_title_save_note, note_draft)
        messages.append({"role": "user", "content": [{
            "type":"tool_result",
            "tool_use_id": tool_block.id,
            "content": save_note_res
            }]
        })

    # get_full_jd tool request
    elif job_title_get_full_jd != None:
        print(f"Iteration {i}, calling get_full_jd \n")
        jd_res = get_full_jd(job_title_get_full_jd)

        messages.append({"role": "user", "content": [{
            "type": "tool_result",
            "tool_use_id": tool_block.id,
            "content": jd_res
            }]
        })

    # get_profile tool request
    elif need_profile:
        print(f"Iteration {i}, calling get_profile \n")
        profile_res = get_profile()
        messages.append({"role": "user", "content": [{
            "type": "tool_result",
            "tool_use_id": tool_block.id,
            "content": profile_res
            }]
        })

    # Unknown tool request
    else:
        messages.append({"role": "user", "content": [{
            "type": "tool_result",
            "tool_use_id": tool_block.id,
            "content": "You called an unknown tool. Double check that you're calling an available tool!"
            }]
        })

    result = client.messages.create(
        model = "claude-haiku-4-5-20251001",
        max_tokens = 1024,
        messages = messages,
        system = SYSTEM_PROMPT,
        tools = tools,
        tool_choice = {"type": "auto", "disable_parallel_tool_use": True}
    )

    print(f"Iteration{i}, Claude's result: ", result.content)
    i += 1

final_ans = ""
print("Final Response: \n")
for block in result.content:
    if block.type == "text":
        final_ans = block
        print(final_ans)

Response before entering the tol_use loop: Message(id='msg_011CfSfR5yxT2pzCpTxo1C5R', container=None, content=[TextBlock(citations=None, text="I'll help you identify which jobs are the best fit! Let me start by looking at your profile to understand your skills and background.", type='text'), ToolUseBlock(id='toolu_01RvrRJNAzmsW5hQRZ4GE1FW', caller=DirectCaller(type='direct'), input={}, name='get_profile', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1261, output_tokens=55, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None)
Iteration 1, no. of input tokens: 1261
Iteration 1, calling get_profile 

Iteration1, Claude's result:  [TextBlock(citat

In [29]:
function_dict = {"get_job_info": get_job_info, "get_profile": get_profile, "save_note": save_note, "get_full_jd": get_full_jd}

print(function_dict["get_job_info"]("python"))


{'Back Bay Life Science Advisors IB Summer Analyst.txt': [], 'DoorDash SWE.txt': ['with at least 1 object-oriented programming language (e.g., python, java, kotlin, etc.)\nyou have experience writing clean'], 'Elire Summer AI Software Development Consulting Intern.txt': [], 'Honeywell Technologies, SWE Co-Op.txt': [], 'Marvell Design for Test Intern, MS.txt': ['overage\nsupport dft flow automation through scripting (tcl, python, or shell) to improve design flows\ncontribute to dft d', 's\nexperience in programming and scripting languages such as python, shell, tcl\nknowledge of using ai assisted tools for c'], 'PIMCO Tech Analyst, SWE.txt': [' to completion \n\npossess strong programming skills, such as python, java, c#, c++, sql, rdbms, and data warehouse skills '], 'Qualcomm Silicon Validation Internship.txt': ['ummer work, or others)\u202f\u202f \nknowledge of scripting languages (python, perl, tcl, shell)\u202f\u202f \nstrong problem-solving skills\u202f \n', 'demic experience with

The next chunk replaced per-tool branching with a lookup table to make the agentic loop shorter
Use this from now on:
- Storing functions in a dictionary, and calling the right one via tool_block.name as the key.
- Extracting the function's input parameters and passing them in using **. BUT ** only works because the schema field names match the parameter names exactly.


In [ ]:
def get_job_info(keyword: str):
    keyword = keyword.lower() # normalize the keyword
    folder = Path("postings") # get filepath to the postings
    results = {}

    for path in folder.glob("*.txt"):
        results[path.name] = []
        text = path.read_text(encoding="utf-8").lower()
        i = 0
        end = len(text)
        # find all instances of the keyword
        while i != -1:
            pos = text.find(keyword, i, end)
            if pos != -1:
                if pos >= 60:
                    results[path.name].append(text[pos-60:pos+61])
                else:
                    results[path.name].append(text[0:pos+61])
                i = pos + len(keyword)
            else: i = -1
    return str(results)

def get_profile():
    # encoding = "utf-8" tells Python which rule to use when turning the file's bytes into characters. UTF-8 is the standard rule modern text uses.
    # It covers every character in every language, including special characters like ● and curly quotes.
    f = open("profile.md", encoding="utf-8")
    profile = f.read()
    f.close()
    return profile

def save_note(job_title: str, note_draft: str):
    print("\n")
    print(job_title)
    print(note_draft)
    return "Draft note printed successfully!"

def get_full_jd(job_title: str):
    path = "Postings/"+ job_title + ".txt"
    f = open(path, encoding="utf-8")
    jd = f.read()
    f.close()
    return jd

tools = [
    {
        "name": "get_job_info",
        "description": "Search all of the user's saved job descriptions to get information about a particular skill or qualification needed for them. Ex: Needing to know PyTorch, Kubernetes, C++, etc. Using this tool will return a dictionary where the key is the filename and the value is a list of snippets containing the keyword from the JD. Note that 1 list corresponds to 1 job description, and a list can have 0, 1 or more snippets in it.  If a key has an empty list, this indicates that the keyword was not in the JD.",
        "input_schema": {
            "type": "object",
            "properties": {
                "keyword": {
                    "type": "string",
                    "description": "The skill/qualification you want to learn more about. Note, this tool only returns things based on keyword search, so you'll have to be specific about what you want. For instance, you cannot generalize and ask about AI-based skills and expect to get back an answer that talks about knowing LangGraph, how to build harnesses, etc. This is one case insensitive keyword/phrase per call. You'll get back matches or no matches with the keyword \"AI skills\". Example input for tool use: PyTorch. ",
                },
            },
            "required": ["keyword"],
        },
    },

    {
        "name": "get_profile",
        "description": "Use this tool to view the user's profile: this includes her skill set, projects, timelines she's looking for, constraints/notes you should know, etc.",
        "input_schema": {
            "type": "object",
            "properties": {}
        },
    },

    {
        "name": "get_full_jd",
        "description": "Get the full job description of jobs you think the user is best qualified for.",
        "input_schema": {
            "type":"object",
            "properties":{
                "job_title": {
                    "type": "string",
                    "description": "Provide the title the EXACT SAME way it was listed in the get_job_info tool responses. Ex: DoorDash SWE"
                },
            },
            "required": ["job_title"]
        },
    },

    {
        "name": "save_note",
        "description": "Use this tool to return a first-draft message (brief cover note/outreach message) to the employer, grounded in information from the user's profile. This tool will print your first draft. You must title it based on the job posting the note is for.",
        "input_schema": {
            "type": "object",
            "properties":{
                "job_title":{
                    "type": "string",
                    "description": "the job posting that the draft-note is for. Ex: PIMCO Tech Analyst, SWE Draft Note."
                },
                "note_draft":{
                    "type": "string",
                    "description": "Add the actual note you've drafted here. Nothing more, nothing else"
                },
            },
            "required": ["job_title", "note_draft"]
        }
    }
]

client = Anthropic(
    api_key = os.environ.get("ANTHROPIC_API_KEY")
)

function_dict = {"get_job_info": get_job_info, "get_profile": get_profile, "save_note": save_note, "get_full_jd": get_full_jd}
MAX_TOOL_USE_ITERATIONS = 2
SYSTEM_PROMPT = "Your role is to help the user find jobs to apply to. Some of the tools provided give you access to (1) the user's profile (what they're looking for, experiences/projects, constraints, etc.) and (2) the job descriptions - these can only be viewed by keyword search to identify specific skills. To best do this job, first look at the user's profile and compare it with skills listed in the job descriptions. Once you get a feel for what the user is and isn't qualified for (based on projects, coursework, skills, etc), and identify the best suited jobs to apply for, you can request to see entire job descriptions to draft outreach note to the recruiter. All of the judgements, statements (including in the outreach message), and assumptions you make must be grounded in the user's profile and what you see in the job descriptions. Note that the output token limit for each turn is 1024 tokens, so keep that in mind as you're writing your final responses. You cannot pause to ask me questions because you're being run in a while stop_reason = tool_use loop."
messages = [{"role": "user", "content": "I have a bunch of job descriptions saved but need some help figuring out which ones I'm most qualified for. You have access to my profile via tool calls, so first go through it, find some skills that I have (Ex: Python, C++, MCP), and compare it with skills listed in the job descriptions. Based on the information you've gathered, tell me which jobs I'd be a good fit for and which I don't know enough for."}]

result = client.messages.create(
    model = "claude-haiku-4-5-20251001",
    max_tokens = 1024,
    system = SYSTEM_PROMPT,
    messages = messages,
    tools = tools,
    tool_choice = {"type": "auto", "disable_parallel_tool_use": True}
)

print("Response before entering the tool_use loop:", result)
i = 0

while result.stop_reason == "tool_use" and i<MAX_TOOL_USE_ITERATIONS:

    # append Claude's request to the message list
    messages.append({"role": "assistant", "content": result.content})
    print(f"Iteration {i}, no. of input tokens: {result.usage.input_tokens}")

    # Processing the tool_block and setting relevant parameters
    for block in result.content:
        if block.type == "tool_use":
            tool_block = block
            break

    # calling the appropriate function
    if tool_block.name in function_dict:
        args = tool_block.input
        # ** in front of a dict, inside a function call, means: "take each key-value pair out of this dict and pass it as key=value
        tool_res = function_dict[tool_block.name](**args)
        print(f"Iteration {i}, calling {tool_block.name} function with the the following parameters: {args}.")

        if i == MAX_TOOL_USE_ITERATIONS-1:
            messages.append({"role": "user",
            "content": [{
                "type": "tool_result",
                "tool_use_id": tool_block.id,
                "content": tool_res + " You've used up all your tool use turns, so you'll have to synthesize the final answer with the info you have. Make it known to the user that you ran out of tool_calls/didn't finish your exploration and are reporting with incomplete information (if that truly is the case)."
                }]
            })

        else:
            messages.append({"role":"user",
                                "content": [{
                                    "type": "tool_result",
                                    "tool_use_id": tool_block.id,
                                    "content": tool_res
                                }]
                            })

    else:
        print(f"Iteration {i}, Claude requested an unknown tool: ", tool_block.name)

        if i == MAX_TOOL_USE_ITERATIONS-1:
            messages.append({"role": "user", "content": [{
                        "type": "tool_result",
                        "tool_use_id": tool_block.id,
                        "content": "You called an unknown tool. Double check that you're calling an available tool! You've also used up all your tool use turns, so you'll have to synthesize the final answer with the info you have. Make it known to the user that you didn't finish your exploration and are reporting with incomplete information if that is actually the case."
                        }]
                    })

        else:
            messages.append({"role": "user", "content": [{
                "type": "tool_result",
                "tool_use_id": tool_block.id,
                "content": "You called an unknown tool. Double check that you're calling an available tool!"
                }]
            })

    result = client.messages.create(
        model = "claude-haiku-4-5-20251001",
        max_tokens = 1024,
        messages = messages,
        system = SYSTEM_PROMPT,
        tools = tools,
        tool_choice = {"type": "auto", "disable_parallel_tool_use": True}
    )

    print(f"Iteration {i}, Claude's result: ", result.content)
    i += 1

final_ans = ""
for block in result.content:
    if block.type == "text":
        final_ans = block
        print(final_ans)

Response before entering the tol_use loop: Message(id='msg_011CfSkUuNBTsXx3rAEz6wnJ', container=None, content=[TextBlock(citations=None, text="I'll help you identify which jobs are the best fit for you. Let me start by looking at your profile to understand your skills and experience.", type='text'), ToolUseBlock(id='toolu_0157HJE83rtseWdMbH5bmtFA', caller=DirectCaller(type='direct'), input={}, name='get_profile', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=1261, output_tokens=57, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None)
Iteration 0, no. of input tokens: 1261
Iteration 0, calling get_profile function with the the following paramete